# 🎬 FILM AGENT — Cloud GPU Studio (Wan 2.2, LTX-2.5 & MiniMax Remote Engine)
### (Google Drive Permanent Model Storage + Instant Symlink Startup + Cloudflare HTTPS Tunnel)

This notebook mounts your **Google Drive**, locates your downloaded models inside **`My Drive › Rise Marketing (Do not delete) › Dilshan › AI_Models`**, links all 4 model folders (**`Wan2.2-I2V-A14B-Diffusers`**, **`Wan2.2-T2V-A14B-Diffusers`**, **`LTX-2.5`**, **`MiniMax-H3`**) in seconds without re-downloading, loads the selected Video Pipeline onto the Google Colab GPU, and exposes an **Async Non-Blocking Public HTTPS Endpoint (`COMFYUI_URL`)** (immune to Cloudflare 524 timeouts) for your local `d:\film-argent` application.

---
### 📌 Quick Start Guide:
1. Upload `video_gen_colab_gpu_studio.ipynb` to Google Colab.
2. Go to **Runtime ➔ Change runtime type ➔ GPU (A100 / L4 / T4)**.
3. Run **Cell 1** to mount Google Drive and verify your 4 model folders inside `Rise Marketing (Do not delete)/Dilshan/AI_Models`.
4. *(Optional)* Run **Cell 2** only if you want to repair or complete any missing Diffusers files (e.g., if `LTX-2.5` only has repackaged weights without `model_index.json`).
5. Run **Cell 3, Cell 4, and Cell 5** in order, then copy the generated **`https://xxxx.trycloudflare.com`** URL into your local Film Agent UI (**Card 0 ➔ Google Colab Video GPU URL**).

In [ ]:
# @title 1️⃣ Cell 1: Mount Google Drive & Verify Models in `Rise Marketing (Do not delete)/Dilshan/AI_Models`
import os
import time
from pathlib import Path
from google.colab import drive

DRIVE_MODELS_ROOT = "/content/drive/MyDrive/Rise Marketing (Do not delete)/Dilshan/AI_Models"  # @param {type:"string"}

t0 = time.time()
MOUNT_POINT = '/content/drive'
if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT, force_remount=False)
print(f'✅ Google Drive mounted in {time.time() - t0:.1f}s!')

def resolve_ai_models_dir(preferred_path: str) -> Path:
    candidates = [
        Path(preferred_path),
        Path('/content/drive/MyDrive/Rise Marketing (Do not delete)/Dilshan/AI_Models'),
        Path('/content/drive/MyDrive/Rise Marketing (Do not delete)/Dilshan/ai models'),
        Path('/content/drive/MyDrive/Rise Marketing (Do not delete)/Dilshan/ai_models'),
        Path('/content/drive/MyDrive/Rise Marketing (Do not delete)/Dilshan'),
        Path('/content/drive/Shareddrives/Rise Marketing (Do not delete)/Dilshan/AI_Models'),
        Path('/content/drive/Shareddrives/Rise Marketing (Do not delete)/Dilshan/ai models'),
        Path('/content/drive/Shareddrives/Rise Marketing (Do not delete)/Dilshan'),
    ]
    for cand in candidates:
        if cand.exists() and cand.is_dir():
            subnames = [p.name.lower() for p in cand.iterdir() if p.is_dir()]
            if any(any(k in s for k in ('wan2.2', 'ltx', 'minimax')) for s in subnames):
                return cand
            for child in cand.iterdir():
                if child.is_dir() and child.name.lower().replace('_', ' ').strip() == 'ai models':
                    return child
    for cand in candidates:
        if cand.exists():
            return cand
    return Path('/content/drive/MyDrive')

ACTIVE_MODELS_ROOT = resolve_ai_models_dir(DRIVE_MODELS_ROOT)
print(f'\n📂 Active AI Models Directory: {ACTIVE_MODELS_ROOT}\n')
print(f"{'Model Folder':<35} | {'Files':>8} | {'Safetensors':>12} | {'Size (GB)':>12} | {'Status'}")
print('-' * 92)

DISCOVERED_MODELS = {
    'wan_i2v': None,
    'wan_t2v': None,
    'ltx': None,
    'minimax': None,
}

total_gb = 0.0
if ACTIVE_MODELS_ROOT.exists():
    for entry in sorted(ACTIVE_MODELS_ROOT.iterdir(), key=lambda x: x.name.lower()):
        if not entry.is_dir() or entry.name.startswith('.'):
            continue
        file_count = 0
        st_count = 0
        folder_bytes = 0
        has_model_index = (entry / 'model_index.json').exists()
        for r, _, files in os.walk(entry):
            if '.cache' in r or '._____temp' in r:
                continue
            for f in files:
                fp = os.path.join(r, f)
                try:
                    folder_bytes += os.path.getsize(fp)
                    file_count += 1
                    if f.lower().endswith('.safetensors'):
                        st_count += 1
                except OSError:
                    pass
        gb = folder_bytes / (1024 ** 3)
        total_gb += gb
        lname = entry.name.lower()
        status = 'Ready'
        if 'wan' in lname and 'i2v' in lname:
            DISCOVERED_MODELS['wan_i2v'] = entry
            status = '✅ Primary I2V Diffusers' if has_model_index else '⚠️ Missing model_index.json'
        elif 'wan' in lname and 't2v' in lname:
            DISCOVERED_MODELS['wan_t2v'] = entry
            status = '✅ Primary T2V Diffusers' if has_model_index else '⚠️ Missing model_index.json'
        elif 'ltx' in lname:
            DISCOVERED_MODELS['ltx'] = entry
            status = '✅ Diffusers Ready' if has_model_index else 'ℹ️ Repackaged Weights (See Cell 2)'
        elif 'minimax' in lname:
            DISCOVERED_MODELS['minimax'] = entry
            status = '✅ Heavy Checkpoints'
        print(f'{entry.name:<35} | {file_count:>8} | {st_count:>12} | {gb:>11.2f} | {status}')

print('-' * 92)
print(f'📦 Total Model Storage Verified in {ACTIVE_MODELS_ROOT.name}: {total_gb:.2f} GB')

In [ ]:
# @title 2️⃣ Cell 2 (Optional): Complete or Repair Any Missing Model Files in `AI_Models`
# Skip this cell if you are using Wan2.2-I2V-A14B-Diffusers (117.54 GB) or Wan2.2-T2V-A14B-Diffusers (117.53 GB),
# as both are already 100% complete in your Google Drive!
# Enable COMPLETE_LTX_DIFFUSERS = True ONLY if you want to add full Diffusers pipeline files to your LTX-2.5 folder.

COMPLETE_LTX_DIFFUSERS = False  # @param {type:"boolean"}

import os
from pathlib import Path

if COMPLETE_LTX_DIFFUSERS:
    !pip install -q -U huggingface_hub hf_transfer
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    from huggingface_hub import snapshot_download

    ltx_dir = DISCOVERED_MODELS.get('ltx') or (ACTIVE_MODELS_ROOT / 'LTX-2.5')
    ltx_dir = Path(ltx_dir)
    ltx_dir.mkdir(parents=True, exist_ok=True)
    print(f'🚀 Completing Diffusers pipeline structure inside {ltx_dir}...')
    snapshot_download(
        repo_id='Lightricks/LTX-Video',
        local_dir=str(ltx_dir),
        ignore_patterns=['*.mp4', '*.gif', '*.png'],
    )
    print(f'✅ LTX Diffusers pipeline is now complete inside: {ltx_dir}')
else:
    print('ℹ️ Skipping download — using existing models in Rise Marketing (Do not delete)/Dilshan/AI_Models.')

In [ ]:
# @title 3️⃣ Cell 3: Install Libraries & Create Instant Symlinks from `Dilshan/AI_Models` (~30 Seconds)
!nvidia-smi
!pip install -q "diffusers>=0.32.0" "transformers>=4.46.0" "accelerate>=0.34.0" safetensors fastapi uvicorn python-multipart imageio imageio-ffmpeg edge-tts opencv-python-headless sentencepiece ftfy

import os
import subprocess
from pathlib import Path

# Install Cloudflared binary for free public HTTPS tunnel (no token required)
if not Path('/usr/local/bin/cloudflared').exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# Create fast local symlink directory (/content/linked_models -> Google Drive/Rise Marketing (Do not delete)/Dilshan/AI_Models)
LINKED_DIR = Path('/content/linked_models')
LINKED_DIR.mkdir(parents=True, exist_ok=True)

linked_items = []
if ACTIVE_MODELS_ROOT.exists():
    for item in sorted(ACTIVE_MODELS_ROOT.iterdir(), key=lambda x: x.name.lower()):
        if item.is_dir() and not item.name.startswith('.'):
            dest = LINKED_DIR / item.name
            if dest.exists() or dest.is_symlink():
                dest.unlink()
            os.symlink(item, dest)
            linked_items.append((item.name, str(item)))

print('✅ All required libraries and Cloudflared tunnel binary are ready!')
print(f'🔗 Instant Symlinks created in {LINKED_DIR}:')
for name, src in linked_items:
    print(f'   • /content/linked_models/{name} ➔ {src}')

In [ ]:
# @title 4️⃣ Cell 4: Load Selected Video Model from `Dilshan/AI_Models` onto Colab GPU
# Choose 'Auto-Detect (Prefer Wan2.2-I2V-A14B)' for Image-to-Video generation with your Master Scene Image,
# or explicitly select Wan2.2-I2V-A14B, Wan2.2-T2V-A14B, or LTX-2.5.

MODEL_ENGINE = "Auto-Detect (Prefer Wan2.2-I2V-A14B-Diffusers)"  # @param ["Auto-Detect (Prefer Wan2.2-I2V-A14B-Diffusers)", "Wan2.2-I2V-A14B-Diffusers (Image-to-Video 14B)", "Wan2.2-T2V-A14B-Diffusers (Text-to-Video 14B)", "LTX-2.5 / LTX-Video"]
CUSTOM_MODEL_PATH = ""  # @param {type:"string"}
ENABLE_FP8_MEMORY_OPTIMIZATION = False  # @param {type:"boolean"}

import gc
import os
import torch
from pathlib import Path

ACTIVE_PIPE = None
ACTIVE_ENGINE_NAME = None
ACTIVE_MODE = 'i2v'  # 'i2v' or 't2v'

def select_model_folder(engine_choice: str) -> Path:
    if CUSTOM_MODEL_PATH and Path(CUSTOM_MODEL_PATH).exists():
        return Path(CUSTOM_MODEL_PATH)

    wan_i2v = DISCOVERED_MODELS.get('wan_i2v') or (ACTIVE_MODELS_ROOT / 'Wan2.2-I2V-A14B-Diffusers')
    wan_t2v = DISCOVERED_MODELS.get('wan_t2v') or (ACTIVE_MODELS_ROOT / 'Wan2.2-T2V-A14B-Diffusers')
    ltx_dir = DISCOVERED_MODELS.get('ltx') or (ACTIVE_MODELS_ROOT / 'LTX-2.5')

    if 'T2V' in engine_choice and Path(wan_t2v).exists():
        return Path(wan_t2v)
    if 'LTX' in engine_choice and Path(ltx_dir).exists():
        return Path(ltx_dir)
    if Path(wan_i2v).exists():
        return Path(wan_i2v)
    if Path(wan_t2v).exists():
        return Path(wan_t2v)
    if Path(ltx_dir).exists():
        return Path(ltx_dir)
    return None

model_path = select_model_folder(MODEL_ENGINE)
if model_path is None or not model_path.exists():
    raise FileNotFoundError(
        f'❌ Could not find model folder in {ACTIVE_MODELS_ROOT}. Please check Cell 1 output!'
    )

symlink_candidate = Path('/content/linked_models') / model_path.name
if symlink_candidate.exists():
    model_path = symlink_candidate

print(f'🚀 Loading Video Model from: {model_path}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0
use_fp8 = ENABLE_FP8_MEMORY_OPTIMIZATION or (vram_gb < 45.0)
print(f'🖥️ Colab GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM) | Precision: {"FP8 Memory Mode" if use_fp8 else "Full 16-Bit Native BFloat16 + Float32 VAE (Ultra-Cinema)"}')

lname = model_path.name.lower()

if 'wan' in lname and 't2v' in lname:
    from diffusers import WanPipeline, AutoencoderKLWan, UniPCMultistepScheduler
    vae = AutoencoderKLWan.from_pretrained(str(model_path), subfolder='vae', torch_dtype=torch.float32)
    ACTIVE_PIPE = WanPipeline.from_pretrained(
        str(model_path),
        vae=vae,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    ACTIVE_PIPE.scheduler = UniPCMultistepScheduler.from_config(ACTIVE_PIPE.scheduler.config, flow_shift=5.0)
    if hasattr(ACTIVE_PIPE.vae, 'enable_tiling'):
        ACTIVE_PIPE.vae.enable_tiling()
    if use_fp8:
        for attr in ('transformer', 'transformer_2'):
            comp = getattr(ACTIVE_PIPE, attr, None)
            if comp is not None:
                try:
                    comp.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16)
                except Exception as e:
                    print(f'FP8 cast notice ({attr}): {e}')
    if vram_gb >= 65.0 and use_fp8:
        ACTIVE_PIPE.to('cuda')
    else:
        ACTIVE_PIPE.enable_model_cpu_offload()
    ACTIVE_ENGINE_NAME = 'Wan2.2-T2V-A14B (Colab GPU)'
    ACTIVE_MODE = 't2v'

elif 'wan' in lname or (model_path / 'transformer_2').exists():
    from diffusers import WanImageToVideoPipeline, AutoencoderKLWan, UniPCMultistepScheduler
    vae = AutoencoderKLWan.from_pretrained(str(model_path), subfolder='vae', torch_dtype=torch.float32)
    ACTIVE_PIPE = WanImageToVideoPipeline.from_pretrained(
        str(model_path),
        vae=vae,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    ACTIVE_PIPE.scheduler = UniPCMultistepScheduler.from_config(ACTIVE_PIPE.scheduler.config, flow_shift=5.0)
    if hasattr(ACTIVE_PIPE.vae, 'enable_tiling'):
        ACTIVE_PIPE.vae.enable_tiling()
    if use_fp8:
        for attr in ('transformer', 'transformer_2'):
            comp = getattr(ACTIVE_PIPE, attr, None)
            if comp is not None:
                try:
                    comp.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16)
                except Exception as e:
                    print(f'FP8 cast notice ({attr}): {e}')
    if vram_gb >= 65.0 and use_fp8:
        ACTIVE_PIPE.to('cuda')
    else:
        ACTIVE_PIPE.enable_model_cpu_offload()
    ACTIVE_ENGINE_NAME = 'Wan2.2-I2V-A14B (Colab GPU)'
    ACTIVE_MODE = 'i2v'

else:
    from diffusers import LTXImageToVideoPipeline
    load_source = str(model_path) if (model_path / 'model_index.json').exists() else 'Lightricks/LTX-Video'
    if load_source != str(model_path):
        print(f'ℹ️ Note: {model_path.name} contains repackaged weights without model_index.json; loading LTXImageToVideoPipeline base components...')
    ACTIVE_PIPE = LTXImageToVideoPipeline.from_pretrained(
        load_source,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    if use_fp8 and hasattr(ACTIVE_PIPE, 'transformer') and ACTIVE_PIPE.transformer is not None:
        try:
            ACTIVE_PIPE.transformer.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16)
        except Exception as e:
            print(f'FP8 cast notice (LTX transformer): {e}')
    if hasattr(ACTIVE_PIPE.vae, 'enable_tiling'):
        ACTIVE_PIPE.vae.enable_tiling()
    ACTIVE_PIPE.enable_model_cpu_offload()
    ACTIVE_ENGINE_NAME = f'LTX-Video ({model_path.name} on Colab GPU)'
    ACTIVE_MODE = 'i2v'

gc.collect()
torch.cuda.empty_cache()
print(f'✅ {ACTIVE_ENGINE_NAME} [mode={ACTIVE_MODE.upper()}] loaded and ready for inference!')

In [ ]:
# @title 5️⃣ Cell 5: Start Async Non-Blocking HTTPS Server & Cloudflare Tunnel (`COMFYUI_URL`)
# Runs GPU video generation in a background worker thread and supports instant async job polling
# so Cloudflare's 100-second timeout (HTTP 524) can NEVER interrupt your video generation!

import io
import re
import time
import uuid
import subprocess
import threading
from pathlib import Path
import torch
import imageio
import numpy as np
from PIL import Image
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, Query
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse

OUT_VIDEOS_DIR = Path('/content/generated_videos')
OUT_VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title='Film Agent — Cloud GPU Video Studio (Wan 2.2 & LTX-2.5)')
GPU_LOCK = threading.Lock()
JOBS = {}
JOB_STATUS = {
    'job_id': None,
    'step': 0,
    'total_steps': 30,
    'status': 'idle',
    'error': None,
    'last_video': None,
}

@app.get('/', response_class=HTMLResponse)
def index_ui():
    return f"""
    <html><head><title>Film Agent Cloud GPU Studio</title>
    <style>body{{background:#0f172a;color:#f8fafc;font-family:sans-serif;max-width:720px;margin:40px auto;padding:20px}}
    .card{{background:#1e293b;padding:24px;border-radius:12px;border:1px solid #334155}}
    input,textarea,button{{width:100%;margin-top:10px;padding:10px;border-radius:8px;border:1px solid #475569;background:#0f172a;color:#fff}}
    button{{background:#10b981;color:#052e16;font-weight:bold;cursor:pointer;border:none;padding:14px}}</style></head>
    <body><div class='card'>
    <h2>🎬 {ACTIVE_ENGINE_NAME} — Ultra-Cinema 720p HD Studio</h2>
    <p>Status: <b>ONLINE</b> | GPU: <b>{torch.cuda.get_device_name(0)}</b> | Source: <b>{ACTIVE_MODELS_ROOT}</b></p>
    <p>Current Job: <b>{JOB_STATUS['status']}</b> (Step {JOB_STATUS['step']}/{JOB_STATUS['total_steps']})</p>
    <form action='/api/v1/generate-i2v' method='post' enctype='multipart/form-data'>
      <label>Master Keyframe Image (PNG/JPG):</label>
      <input type='file' name='image' required />
      <label>Motion Prompt:</label>
      <textarea name='prompt' rows='3'>Medium close-up studio video of a young South Asian woman speaking naturally with warm facial expression, realistic lip movement, natural eye blinks, steady cinema camera.</textarea>
      <label>Resolution (Width x Height):</label>
      <input type='number' name='width' value='1280' />
      <input type='number' name='height' value='720' />
      <label>Frames (81 = 5.0s native motion):</label>
      <input type='number' name='num_frames' value='81' />
      <label>Inference Steps (30 = Ultra-Cinema Detail):</label>
      <input type='number' name='steps' value='30' />
      <input type='hidden' name='async_job' value='0' />
      <button type='submit'>🚀 Generate 720p HD Video (.mp4)</button>
    </form></div></body></html>
    """

@app.get('/health')
def health_check(job_id: str = Query(None)):
    active_job = JOBS.get(job_id) if (job_id and job_id in JOBS) else JOB_STATUS
    return {
        'status': 'ok',
        'engine': ACTIVE_ENGINE_NAME,
        'mode': ACTIVE_MODE,
        'models_root': str(ACTIVE_MODELS_ROOT),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'job': active_job,
    }

@app.get('/api/v1/download-video')
def download_video_endpoint(job_id: str = Query(None)):
    target_job = JOBS.get(job_id) if (job_id and job_id in JOBS) else JOB_STATUS
    video_path = target_job.get('last_video') or JOB_STATUS.get('last_video')
    if not video_path or not Path(video_path).exists():
        # Also check most recent MP4 in OUT_VIDEOS_DIR as fallback
        mp4s = sorted(OUT_VIDEOS_DIR.glob('*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)
        if mp4s:
            video_path = str(mp4s[0])
        else:
            return JSONResponse({'error': 'Video not found or still rendering'}, status_code=404)
    vp = Path(video_path)
    return FileResponse(str(vp), media_type='video/mp4', filename=vp.name)

def _execute_gpu_generation(job_id, pil_img, prompt, negative_prompt, width, height, num_frames, steps, fps, wan_model):
    with GPU_LOCK:
        try:
            # Dynamically tune Wan 2.2 scheduler flow_shift (5.0 for 720p HD, 3.0 for 480p) and VAE tiling
            if 'Wan' in str(ACTIVE_ENGINE_NAME):
                try:
                    from diffusers import UniPCMultistepScheduler
                    target_shift = 5.0 if max(width, height) >= 960 else 3.0
                    ACTIVE_PIPE.scheduler = UniPCMultistepScheduler.from_config(ACTIVE_PIPE.scheduler.config, flow_shift=target_shift)
                except Exception as sched_err:
                    print(f'Scheduler tuning notice: {sched_err}')
            if hasattr(ACTIVE_PIPE, 'vae') and hasattr(ACTIVE_PIPE.vae, 'enable_tiling'):
                ACTIVE_PIPE.vae.enable_tiling()

            print(f'[Job {job_id} Start] Engine={ACTIVE_ENGINE_NAME} | Requested={wan_model} | {width}x{height} HD | {num_frames}f | {steps} steps | target_fps={fps}')
            JOBS[job_id].update({'step': 0, 'total_steps': steps, 'status': 'running', 'error': None})
            JOB_STATUS.update(JOBS[job_id])

            def _cb(pipe, step_idx, timestep, kwargs):
                cur = step_idx + 1
                JOBS[job_id]['step'] = cur
                JOB_STATUS['step'] = cur
                print(f'   [Job {job_id}] Step {cur}/{steps} ({int(cur * 100 / max(steps, 1))}%)')
                return kwargs

            with torch.inference_mode():
                if ACTIVE_MODE == 't2v':
                    out = ACTIVE_PIPE(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        width=width,
                        height=height,
                        num_frames=num_frames,
                        num_inference_steps=steps,
                        guidance_scale=5.0,
                        callback_on_step_end=_cb,
                    )
                else:
                    out = ACTIVE_PIPE(
                        image=pil_img,
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        width=width,
                        height=height,
                        num_frames=num_frames,
                        num_inference_steps=steps,
                        guidance_scale=5.0 if 'Wan' in str(ACTIVE_ENGINE_NAME) else 3.2,
                        callback_on_step_end=_cb,
                    )

            frames = out.frames[0]
            out_path = OUT_VIDEOS_DIR / f'i2v_{int(time.time())}_{job_id}.mp4'
            # Wan 2.2 generates 81 frames at a native 16fps cadence (5.06 seconds).
            # Encode at native 16fps first with lossless-grade CRF 14, then resample to 24fps while preserving exact 5.06s duration.
            native_fps = 16 if 'Wan' in str(ACTIVE_ENGINE_NAME) else fps
            raw_path = OUT_VIDEOS_DIR / f'raw_{job_id}.mp4' if fps != native_fps else out_path
            writer = imageio.get_writer(
                str(raw_path),
                fps=native_fps,
                codec='libx264',
                ffmpeg_params=['-crf', '14', '-preset', 'slow', '-pix_fmt', 'yuv420p'],
            )
            for f in frames:
                arr = np.array(f.convert('RGB')) if isinstance(f, Image.Image) else np.array(f)
                writer.append_data(arr)
            writer.close()

            if raw_path != out_path:
                try:
                    subprocess.run(
                        [
                            'ffmpeg', '-y', '-i', str(raw_path),
                            '-vf', f'fps={fps}',
                            '-c:v', 'libx264', '-crf', '14', '-preset', 'slow', '-pix_fmt', 'yuv420p',
                            str(out_path),
                        ],
                        check=True,
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL,
                    )
                    raw_path.unlink(missing_ok=True)
                except Exception:
                    raw_path.rename(out_path)

            JOBS[job_id].update({'step': steps, 'status': 'completed', 'last_video': str(out_path)})
            JOB_STATUS.update(JOBS[job_id])
            print(f'✅ [Job {job_id} Complete] Saved Ultra-Cinema MP4: {out_path} ({out_path.stat().st_size / (1024*1024):.2f} MB)')
        except Exception as e:
            err_msg = str(e)
            print(f'❌ [Job {job_id} Error] {err_msg}')
            JOBS[job_id].update({'status': 'error', 'error': err_msg})
            JOB_STATUS.update(JOBS[job_id])

@app.post('/api/v1/generate-i2v')
async def generate_i2v_endpoint(
    image: UploadFile = File(...),
    prompt: str = Form('Natural cinematic motion, speaking expressively, steady camera.'),
    negative_prompt: str = Form('static frozen image, blurry, morphing, distorted face, warping microphone, worst quality'),
    width: int = Form(1280),
    height: int = Form(720),
    num_frames: int = Form(81),
    steps: int = Form(30),
    fps: int = Form(24),
    wan_model: str = Form('Auto'),
    async_job: str = Form('1'),
):
    width = max(256, (int(width) // 32) * 32)
    height = max(256, (int(height) // 32) * 32)
    num_frames = max(17, ((int(num_frames) - 1) // 4) * 4 + 1)
    steps = max(1, int(steps))
    fps = max(8, int(fps))

    raw_bytes = await image.read()
    pil_img = Image.open(io.BytesIO(raw_bytes)).convert('RGB').resize((width, height), Image.Resampling.LANCZOS)

    job_id = uuid.uuid4().hex[:10]
    JOBS[job_id] = {
        'job_id': job_id,
        'step': 0,
        'total_steps': steps,
        'status': 'running',
        'error': None,
        'last_video': None,
    }
    JOB_STATUS.update(JOBS[job_id])

    worker = threading.Thread(
        target=_execute_gpu_generation,
        args=(job_id, pil_img, prompt, negative_prompt, width, height, num_frames, steps, fps, wan_model),
        daemon=True,
    )
    worker.start()

    # Async Job Mode (Default for Film Agent Backend — returns in 0.1s so Cloudflare never times out with 524!)
    if str(async_job).strip() in ('1', 'true', 'True', 'yes'):
        return JSONResponse({'status': 'running', 'job_id': job_id, 'total_steps': steps}, status_code=202)

    # Synchronous fallback for direct browser form submission
    while JOBS[job_id]['status'] == 'running':
        await asyncio.sleep(1.0)
    if JOBS[job_id]['status'] == 'completed' and JOBS[job_id]['last_video']:
        vp = Path(JOBS[job_id]['last_video'])
        return FileResponse(str(vp), media_type='video/mp4', filename=vp.name)
    return JSONResponse({'error': JOBS[job_id].get('error') or 'Generation failed'}, status_code=500)

def start_tunnel():
    proc = subprocess.Popen(
        ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            print('\n' + '=' * 75)
            print(f'🌐 PUBLIC REMOTE GPU URL : {url}')
            print(f'👉 Paste this into Film Agent UI (Card 0 -> Google Colab Video GPU URL)')
            print(f'👉 Or save in d:\\film-argent\\.env -> COMFYUI_URL={url}')
            print('=' * 75 + '\n')

import asyncio
threading.Thread(target=start_tunnel, daemon=True).start()
config = uvicorn.Config(app, host='0.0.0.0', port=8188, log_level='info')
server = uvicorn.Server(config)
await server.serve()